In [1]:
import os
import numpy as np
import pandas as pd
import joblib

# Random Forest Models
from sklearn.ensemble import RandomForestRegressor
# Evaluation Metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
# Model Tuning
import optuna

/home/aliarridha/miniconda3/envs/modeling1/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Setup path
BASE_PATH = 'dataset'
OUTPUT_FOLDER = os.path.join(BASE_PATH, 'output_pipeline')
ARTIFACTS_FOLDER = os.path.join(BASE_PATH, 'artifacts')
MODEL_FOLDER = os.path.join(BASE_PATH, 'model_outputs')
EDA_FOLDER = os.path.join(BASE_PATH, 'eda_outputs')
RESULTS_FOLDER = os.path.join(BASE_PATH, 'results')

os.makedirs(MODEL_FOLDER, exist_ok=True)
os.makedirs(EDA_FOLDER, exist_ok=True)
os.makedirs(RESULTS_FOLDER, exist_ok=True)

In [3]:
# Load fitur (X)
X_train = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_X_train.parquet'))
X_val = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_X_val.parquet'))
X_test = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_X_test.parquet'))

In [4]:
# Load target (y) - convert ke Series untuk sklearn
y_train = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_y_train.parquet'))
y_val = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_y_val.parquet'))
y_test = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_y_test.parquet'))

In [5]:
# Load metadata ID (untuk traceback hasil prediksi nanti)
ID_train = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_ID_train.parquet'))
ID_val = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_ID_val.parquet'))
ID_test = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_ID_test.parquet'))

In [6]:
scaler = joblib.load(os.path.join(ARTIFACTS_FOLDER, 'minmax_scaler.joblib'))
winsor = joblib.load(os.path.join(ARTIFACTS_FOLDER, 'winsor_bounds.joblib'))
feat_select = joblib.load(os.path.join(ARTIFACTS_FOLDER, 'feature_selection_info.joblib'))

/home/aliarridha/miniconda3/envs/modeling1/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.6.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [7]:
TARGET = feat_select['target']

In [8]:
y_train_arr = y_train[TARGET].values
y_val_arr = y_val[TARGET].values
y_test_arr = y_test[TARGET].values

In [9]:
print(ID_test.head())
print(ID_test.columns)

         kabupaten_kota   nama_wilayah_bersih  tahun
0       Kab. Aceh Barat       kab. aceh barat   2024
1       Kab. Aceh Barat       kab. aceh barat   2025
2  Kab. Aceh Barat Daya  kab. aceh barat daya   2024
3  Kab. Aceh Barat Daya  kab. aceh barat daya   2025
4       Kab. Aceh Besar       kab. aceh besar   2024
Index(['kabupaten_kota', 'nama_wilayah_bersih', 'tahun'], dtype='object')


In [10]:
print("Mulai tuning Random Forest (Bayesian - Optuna)")

def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 400, step=50),
        "max_depth": trial.suggest_int("max_depth", 10, 20),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 10),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 5),
    }

    model = RandomForestRegressor(
        random_state=42,
        n_jobs=-1,
        **params,
    )
    model.fit(X_train, y_train_arr)
    val_pred = model.predict(X_val)
    val_mae = mean_absolute_error(y_val_arr, val_pred)
    return val_mae

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=35, show_progress_bar=True)

best_params = study.best_params
best_mae = study.best_value

rf_model = RandomForestRegressor(
    random_state=42,
    n_jobs=-1,
    **best_params,
)
rf_model.fit(X_train, y_train_arr)

print("Tuning selesai")
print("Best params:")
print(best_params)
print(f"Best Validation MAE: {best_mae:.4f}")

[I 2026-06-04 15:58:21,936] A new study created in memory with name: no-name-42ae4216-a078-44af-9501-108cf91fe7b5


Mulai tuning Random Forest (Bayesian - Optuna)


Best trial: 0. Best value: 4.06912:   3%|▎         | 1/35 [00:01<00:35,  1.05s/it]

[I 2026-06-04 15:58:22,987] Trial 0 finished with value: 4.069123723806911 and parameters: {'n_estimators': 350, 'max_depth': 18, 'min_samples_split': 5, 'min_samples_leaf': 5}. Best is trial 0 with value: 4.069123723806911.


Best trial: 0. Best value: 4.06912:   6%|▌         | 2/35 [00:01<00:31,  1.05it/s]

...

[I 2026-06-04 15:58:48,222] Trial 32 finished with value: 3.9573062406113064 and parameters: {'n_estimators': 200, 'max_depth': 19, 'min_samples_split': 2, 'min_samples_leaf': 1}. Best is trial 6 with value: 3.937423272197413.


Best trial: 6. Best value: 3.93742:  97%|█████████▋| 34/35 [00:26<00:00,  1.28it/s]

[I 2026-06-04 15:58:48,922] Trial 33 finished with value: 3.9448908645715366 and parameters: {'n_estimators': 200, 'max_depth': 20, 'min_samples_split': 2, 'min_samples_leaf': 1}. Best is trial 6 with value: 3.937423272197413.


Best trial: 6. Best value: 3.93742: 100%|██████████| 35/35 [00:27<00:00,  1.25it/s]


[I 2026-06-04 15:58:49,865] Trial 34 finished with value: 3.9501683204150497 and parameters: {'n_estimators': 300, 'max_depth': 18, 'min_samples_split': 4, 'min_samples_leaf': 1}. Best is trial 6 with value: 3.937423272197413.
Tuning selesai
Best params:
{'n_estimators': 350, 'max_depth': 16, 'min_samples_split': 3, 'min_samples_leaf': 1}
Best Validation MAE: 3.9374


In [11]:
# Prediksi
rf_train_pred = rf_model.predict(X_train)
rf_val_pred = rf_model.predict(X_val)
rf_test_pred = rf_model.predict(X_test)

In [12]:
def hitung_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(
        mean_squared_error(y_true, y_pred)
    )
    r2 = r2_score(y_true, y_pred)
    return mae, rmse, r2

In [13]:
# evaluasi train
rf_train_mae, rf_train_rmse, rf_train_r2 = hitung_metrics(
    y_train,
    rf_train_pred
)

In [14]:
# evaluasi validation
rf_val_mae, rf_val_rmse, rf_val_r2 = hitung_metrics(
    y_val,
    rf_val_pred
)

In [15]:
# evaluasi test
rf_test_mae, rf_test_rmse, rf_test_r2 = hitung_metrics(
    y_test,
    rf_test_pred
)

In [16]:
print("HASIL RANDOM FOREST")

print("\nTrain")
print(f"MAE  : {rf_train_mae:.4f}")
print(f"RMSE : {rf_train_rmse:.4f}")
print(f"R2   : {rf_train_r2:.4f}")

print("\nValidation")
print(f"MAE  : {rf_val_mae:.4f}")
print(f"RMSE : {rf_val_rmse:.4f}")
print(f"R2   : {rf_val_r2:.4f}")

print("\nTest")
print(f"MAE  : {rf_test_mae:.4f}")
print(f"RMSE : {rf_test_rmse:.4f}")
print(f"R2   : {rf_test_r2:.4f}")

HASIL RANDOM FOREST

Train
MAE  : 1.8628
RMSE : 2.4129
R2   : 0.9495

Validation
MAE  : 3.9374
RMSE : 5.0488
R2   : 0.7592

Test
MAE  : 4.1610
RMSE : 5.4813
R2   : 0.7304


In [17]:
# Simpan Model Ridge Regression
joblib.dump(
    rf_model,
    os.path.join(BASE_PATH, 'models', 'random_forest.joblib')
)
print("Model berhasil disimpan.")

Model berhasil disimpan.


In [18]:
# Simpan metrics
metrics_rf = pd.DataFrame({
    "model": ["Random Forest"],
    "train_mae": [rf_train_mae],
    "train_rmse": [rf_train_rmse],
    "train_r2": [rf_train_r2],
    "val_mae": [rf_val_mae],
    "val_rmse": [rf_val_rmse],
    "val_r2": [rf_val_r2],
    "test_mae": [rf_test_mae],
    "test_rmse": [rf_test_rmse],
    "test_r2": [rf_test_r2]
})


os.makedirs("results", exist_ok=True)

metrics_rf.to_csv(
    os.path.join(BASE_PATH, "results", "random_forest_metrics.csv"),
    index=False
)
print("Metrics berhasil disimpan.")

Metrics berhasil disimpan.


In [19]:
# Simpan hasil prediksi test
hasil_pred_lr = ID_test.copy()

hasil_pred_lr["actual"] = y_test.values
hasil_pred_lr["prediction_rf"] = rf_test_pred

hasil_pred_lr.to_csv(
    os.path.join(BASE_PATH, "results", "random_forest_predictions.csv"),
    index=False
)

print("Hasil prediksi berhasil disimpan.")

# preview hasil
print("\nPreview prediction:")
print(hasil_pred_lr.head())

Hasil prediksi berhasil disimpan.

Preview prediction:
         kabupaten_kota   nama_wilayah_bersih  tahun  actual  prediction_rf
0       Kab. Aceh Barat       kab. aceh barat   2024   54.82      46.306808
1       Kab. Aceh Barat       kab. aceh barat   2025   53.53      49.687296
2  Kab. Aceh Barat Daya  kab. aceh barat daya   2024   55.15      52.597119
3  Kab. Aceh Barat Daya  kab. aceh barat daya   2025   66.77      53.907688
4       Kab. Aceh Besar       kab. aceh besar   2024   51.82      48.234325


In [20]:
feature_importance_rf = pd.DataFrame({
    "feature": X_train.columns,
    "importance": rf_model.feature_importances_
})

feature_importance_rf = feature_importance_rf.sort_values(
    by="importance",
    ascending=False
)

feature_importance_rf.to_csv(
    os.path.join(BASE_PATH, "results", 'random_forest_feature_importance.csv'),
    index=False
)

print("Hasil RF feature importance berhasil disimpan.")

Hasil RF feature importance berhasil disimpan.


In [22]:
joblib.dump(
    rf_model,
    os.path.join(
        MODEL_FOLDER,
        'random_forest_model.joblib'
    )
)

print("Random Forest model berhasil disimpan.")

Random Forest model berhasil disimpan.
